# Direction Y: Statistical rigor + lineage-aware robustness cho adaptive feature fusion (5 thuốc gốc)


1. **Ý nghĩa thống kê** — lợi thế của feature selection dựa trên accessory genome so với paper-ready 50 markers có thật (significant) hay chỉ là nhiễu? → repeated stratified CV + corrected resampled *t*-test (Nadeau–Bengio) + Wilcoxon signed-rank + bootstrap 95% CI + hiệu chỉnh Holm–Bonferroni trên 5 thuốc.
2. **Lineage leakage** — điểm cao có phải do cấu trúc dòng (population structure) khi dùng random split? → cluster mẫu bằng Jaccard trên accessory genome rồi **leave-cluster-out / GroupKFold**, đo mức tụt hiệu năng so với random split.

Thêm **negative control** (shuffled labels) trên chính 5 thuốc gốc để chứng minh pipeline không học được gì từ nhãn ngẫu nhiên.


## 0. Imports + cấu hình

In [ ]:
import os, re, json, math, shutil, warnings, subprocess, itertools
from pathlib import Path
from collections import Counter, defaultdict

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import display

from scipy import stats as sstats
from scipy.sparse.csgraph import connected_components
from scipy import sparse

from sklearn.model_selection import StratifiedKFold, GroupKFold, train_test_split
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, average_precision_score,
)
import matplotlib.pyplot as plt

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

print("HAS_XGB:", HAS_XGB)

In [ ]:
# ---- Cấu hình ----
REPO_URL = "https://github.com/347251369/Antimicrobial-resistance-prediction-in-Salmonella.git"
BASE_DIR = Path("/content/salmonella_direction_Y_robustness")
REPO_DIR = BASE_DIR / "Antimicrobial-resistance-prediction-in-Salmonella"
EXTRACT_DIR = BASE_DIR / "extracted"
OUTPUT_DIR = BASE_DIR / "outputs"
for d in [BASE_DIR, EXTRACT_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DRUGS = ["AMP", "AUG", "AXO", "CHL", "FOX"]

# Repeated stratified CV
N_REPEATS = 5          # số lần lặp CV
N_SPLITS  = 5          # số fold mỗi lần
BASE_SEED = 42
RANDOM_SEEDS = list(range(BASE_SEED, BASE_SEED + N_REPEATS))

# Feature modules
K_CHI2 = 200                     # top-k accessory theo chi2 (fit chỉ trên train)
MAX_CANDIDATE_FEATURES = 8000    # giới hạn cột accessory đưa vào chi2 cho nhanh

# Model set
MODEL_NAMES = ["LR_balanced", "RF_balanced"] + (["XGB_weighted"] if HAS_XGB else [])

# Lineage-aware
# Lưu ý: accessory genome của bộ này rất đa dạng (Jaccard tối đa giữa 2 mẫu ~0.07),
# nên gộp cụm theo NGƯỠNG sẽ ra toàn cụm đơn lẻ (= random split). Vì vậy ta ép về
# K cụm bằng agglomerative clustering trên khoảng cách Jaccard để leave-cluster-out có nghĩa.
N_LINEAGE_CLUSTERS = 30
ALPHA = 0.05                               # mức ý nghĩa

print("BASE_DIR:", BASE_DIR)
print("MODEL_NAMES:", MODEL_NAMES)
print("CV:", N_REPEATS, "x", N_SPLITS)

## 1. Clone repo + giải nén accessory gene matrix (giống Direction O)

In [ ]:
if not REPO_DIR.exists():
    !git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"
else:
    print("Repo đã tồn tại:", REPO_DIR)

!apt-get update -qq
!apt-get install -y unrar > /dev/null

accessory_extract_dir = EXTRACT_DIR / "accessory_gene"
accessory_extract_dir.mkdir(parents=True, exist_ok=True)
accessory_rar = REPO_DIR / "results" / "Roary" / "accessory gene existence matrix.rar"

if not any(accessory_extract_dir.glob("*")):
    if accessory_rar.exists():
        print("Giải nén:", accessory_rar)
        !unrar x -o+ "{accessory_rar}" "{accessory_extract_dir}/" > /dev/null
    else:
        print("Không thấy RAR trong repo, tải trực tiếp.")
        local_rar = BASE_DIR / "accessory_gene_existence_matrix.rar"
        url = "https://github.com/347251369/Antimicrobial-resistance-prediction-in-Salmonella/raw/main/results/Roary/accessory%20gene%20existence%20matrix.rar"
        !wget -q -O "{local_rar}" "{url}"
        !unrar x -o+ "{local_rar}" "{accessory_extract_dir}/" > /dev/null

!find "{accessory_extract_dir}" -maxdepth 2 -type f | head -20

## 2. Hàm nạp dữ liệu (tái dùng convention của Direction O)

In [ ]:
def list_files(root, suffixes=None):
    root = Path(root); files = []
    for p in root.rglob("*"):
        if p.is_file() and (suffixes is None or p.suffix.lower() in suffixes):
            files.append(p)
    return files

def find_largest_table(root):
    cands = list_files(root, suffixes=[".csv", ".tsv", ".txt", ".xlsx", ".xls"])
    if not cands:
        raise FileNotFoundError(f"Không tìm thấy bảng trong {root}")
    cands = sorted(cands, key=lambda p: p.stat().st_size, reverse=True)
    return cands[0]

def read_table_flexible(path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in [".tsv", ".txt"]:
        df = pd.read_csv(path, sep="\t")
        return pd.read_csv(path) if df.shape[1] == 1 else df
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    raise ValueError(path)

def coerce_numeric_features(df):
    out = df.copy(); drop_cols = []
    for col in list(out.columns):
        if out[col].dtype == "object":
            conv = pd.to_numeric(out[col], errors="coerce")
            if conv.notna().mean() > 0.95:
                out[col] = conv.fillna(0)
            else:
                drop_cols.append(col)
    if drop_cols:
        out = out.drop(columns=drop_cols)
    return out.fillna(0)

def parse_label_series(y_raw):
    y = y_raw.copy()
    if isinstance(y, pd.DataFrame):
        cands = [c for c in y.columns if any(k in c.lower() for k in ["label","phenotype","result","concl"])]
        y = y[cands[0]] if cands else y[y.columns[-1]]
    y = y.replace({"S":0,"I":0,"R":1,"s":0,"i":0,"r":1,
                   "Susceptible":0,"Intermediate":0,"Resistant":1,
                   "susceptible":0,"intermediate":0,"resistant":1})
    return pd.to_numeric(y, errors="coerce")

def find_drug_label_file(drug):
    drug_dir = REPO_DIR / "data" / "csv" / drug
    exact = drug_dir / f"{drug}_label.csv"
    if exact.exists():
        return exact
    cands = list(drug_dir.glob("*label*.csv"))
    if cands:
        return cands[0]
    raise FileNotFoundError(f"Không tìm thấy label csv cho {drug}")

def load_ready_and_mask(drug):
    """Trả về (X_ready, y, positions) với positions = vị trí dòng gốc được giữ (để align accessory)."""
    drug_dir = REPO_DIR / "data" / "csv" / drug
    X = coerce_numeric_features(pd.read_csv(drug_dir / "gene.csv"))
    y_full = parse_label_series(pd.read_csv(find_drug_label_file(drug)))
    mask = y_full.notna().values
    positions = np.where(mask)[0]
    X = X.loc[mask].reset_index(drop=True)
    y = y_full.loc[mask].reset_index(drop=True).astype(int)
    return X, y, positions

In [ ]:
# Nạp accessory matrix 1 lần (toàn bộ 1167 dòng, thứ tự gốc)
accessory_path = find_largest_table(accessory_extract_dir)
X_ACC_FULL = coerce_numeric_features(read_table_flexible(accessory_path))
print("Accessory matrix:", X_ACC_FULL.shape)

# Nạp 5 thuốc
ready = {}
rows = []
for drug in DRUGS:
    Xr, yr, pos = load_ready_and_mask(drug)
    # align accessory theo vị trí dòng gốc được giữ
    if X_ACC_FULL.shape[0] >= (pos.max() + 1):
        Xacc = X_ACC_FULL.iloc[pos].reset_index(drop=True)
    else:
        raise ValueError(f"Accessory rows {X_ACC_FULL.shape[0]} không đủ để align {drug}")
    ready[drug] = {"X_ready": Xr, "X_acc": Xacc, "y": yr}
    rows.append({"drug": drug, "n": len(yr), "ready_feats": Xr.shape[1],
                 "acc_feats": Xacc.shape[1], "n_R": int(yr.sum()),
                 "resistant_rate": round(float(yr.mean()), 4)})
stats_df = pd.DataFrame(rows)
display(stats_df)
stats_df.to_csv(OUTPUT_DIR / "Y_dataset_stats.csv", index=False)

## 3. Feature modules + model + đánh giá 1 fold (không rò rỉ)

Ba module tái lập được (bỏ các module đồ thị/embedding phức tạp để notebook tự chứa):

- `paper_ready50` — baseline (50 marker do bài báo chọn).
- `accessory_chi2_k` — chọn top-k cột accessory bằng chi2, **fit chỉ trên train fold**.
- `hybrid_ready+chi2` — ghép paper-ready 50 với top-k accessory.

Trong mỗi train/test fold: selector fit trên train; tách inner train/val để tune ngưỡng; refit trên toàn bộ train; đánh giá trên test. Test chỉ dùng để report.


In [ ]:
def make_model(name, y_train, seed):
    if name == "LR_balanced":
        return LogisticRegression(max_iter=5000, class_weight="balanced",
                                  solver="lbfgs", random_state=seed)
    if name == "RF_balanced":
        return RandomForestClassifier(n_estimators=400, class_weight="balanced",
                                      n_jobs=-1, random_state=seed)
    if name == "XGB_weighted":
        pos = max(int((y_train == 1).sum()), 1)
        neg = max(int((y_train == 0).sum()), 1)
        return xgb.XGBClassifier(
            n_estimators=400, max_depth=4, learning_rate=0.1,
            subsample=0.9, colsample_bytree=0.9, eval_metric="logloss",
            scale_pos_weight=neg / pos, random_state=seed, n_jobs=-1,
            use_label_encoder=False)
    raise ValueError(name)

def select_chi2_topk(X_acc_tr, y_tr, k):
    Xt = X_acc_tr.values.astype(float)
    var = Xt.var(axis=0)
    keep = np.where(var > 0)[0]
    if len(keep) > MAX_CANDIDATE_FEATURES:
        keep = keep[np.argsort(var[keep])[::-1][:MAX_CANDIDATE_FEATURES]]
    Xk = np.clip(Xt[:, keep], 0, None)
    chi, _ = chi2(Xk, y_tr)
    chi = np.nan_to_num(chi, nan=0.0)
    order = np.argsort(chi)[::-1][:k]
    return keep[order]

def choose_threshold(y_val, prob_val):
    best_t, best = 0.5, -1
    for t in np.linspace(0.05, 0.95, 91):
        s = f1_score(y_val, (prob_val >= t).astype(int), zero_division=0)
        if s > best:
            best, best_t = s, float(t)
    return best_t

def scores(y_true, y_pred, y_prob):
    d = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    try: d["auroc"] = roc_auc_score(y_true, y_prob)
    except Exception: d["auroc"] = np.nan
    try: d["auprc"] = average_precision_score(y_true, y_prob)
    except Exception: d["auprc"] = np.nan
    return d

In [ ]:
def eval_fold(module, model_name, X_ready, X_acc, y, tr_idx, te_idx, seed, k=K_CHI2):
    """Đánh giá 1 fold cho (module, model). Selection + threshold chỉ dùng train."""
    y = np.asarray(y)
    ytr, yte = y[tr_idx], y[te_idx]

    # 1) chọn feature dựa trên train
    if module == "paper_ready50":
        Xtr = X_ready.values[tr_idx]; Xte = X_ready.values[te_idx]
    elif module == "accessory_chi2":
        cols = select_chi2_topk(X_acc.iloc[tr_idx], ytr, k)
        Xtr = X_acc.values[np.ix_(tr_idx, cols)]
        Xte = X_acc.values[np.ix_(te_idx, cols)]
    elif module == "hybrid_ready_chi2":
        cols = select_chi2_topk(X_acc.iloc[tr_idx], ytr, k)
        Xtr = np.hstack([X_ready.values[tr_idx], X_acc.values[np.ix_(tr_idx, cols)]])
        Xte = np.hstack([X_ready.values[te_idx], X_acc.values[np.ix_(te_idx, cols)]])
    else:
        raise ValueError(module)

    Xtr = np.nan_to_num(Xtr.astype(float)); Xte = np.nan_to_num(Xte.astype(float))

    # 2) tune threshold trên inner val
    if len(np.unique(ytr)) < 2:
        return None
    itr, ival = train_test_split(np.arange(len(ytr)), test_size=0.2,
                                 stratify=ytr, random_state=seed)
    m = make_model(model_name, ytr[itr], seed)
    m.fit(Xtr[itr], ytr[itr])
    prob_val = m.predict_proba(Xtr[ival])[:, 1]
    thr = choose_threshold(ytr[ival], prob_val)

    # 3) refit toàn bộ train, đánh giá test
    m = make_model(model_name, ytr, seed)
    m.fit(Xtr, ytr)
    prob_te = m.predict_proba(Xte)[:, 1]
    pred_te = (prob_te >= thr).astype(int)
    return scores(yte, pred_te, prob_te)

## 4. Repeated stratified CV cho mọi (thuốc, module, model)

Thu thập F1 / balanced accuracy / AUPRC theo từng fold để phục vụ kiểm định thống kê ở bước sau.


In [ ]:
MODULES = ["paper_ready50", "accessory_chi2", "hybrid_ready_chi2"]

per_fold_records = []   # từng fold
for drug in DRUGS:
    d = ready[drug]
    X_ready, X_acc, y = d["X_ready"], d["X_acc"], np.asarray(d["y"])
    for rep, seed in enumerate(RANDOM_SEEDS):
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        for fold, (tr_idx, te_idx) in enumerate(skf.split(np.zeros(len(y)), y)):
            for module in MODULES:
                for model_name in MODEL_NAMES:
                    sc = eval_fold(module, model_name, X_ready, X_acc, y,
                                   tr_idx, te_idx, seed=seed + fold)
                    if sc is None:
                        continue
                    per_fold_records.append({
                        "drug": drug, "module": module, "model": model_name,
                        "repeat": rep, "fold": fold, **sc})
    print("Xong CV cho", drug)

fold_df = pd.DataFrame(per_fold_records)
fold_df.to_csv(OUTPUT_DIR / "Y_per_fold_scores.csv", index=False)
print(fold_df.shape)
display(fold_df.head())

In [ ]:
# Bảng trung bình theo (thuốc, module, model)
agg = (fold_df.groupby(["drug", "module", "model"])
       [["f1", "balanced_accuracy", "auprc", "recall", "precision"]]
       .agg(["mean", "std"]))
agg.columns = ["_".join(c) for c in agg.columns]
agg = agg.reset_index()
agg.to_csv(OUTPUT_DIR / "Y_mean_scores.csv", index=False)
display(agg.sort_values(["drug", "f1_mean"], ascending=[True, False]))

## 5. Kiểm định thống kê: adaptive-best vs paper_ready50

Với mỗi thuốc:

- **Baseline** = cấu hình `paper_ready50` có mean F1 cao nhất (chọn model tốt nhất cho baseline).
- **Adaptive-best** = cấu hình accessory/hybrid có mean F1 cao nhất.
- So sánh F1 **theo từng fold ghép cặp** bằng:
  - **Corrected resampled *t*-test (Nadeau–Bengio)** — hiệu chỉnh tương quan giữa các fold của repeated CV.
  - **Wilcoxon signed-rank** — kiểm định phi tham số.
  - **Bootstrap 95% CI** cho mean delta F1.
- **Hiệu chỉnh Holm–Bonferroni** cho 5 phép so sánh (5 thuốc).


In [ ]:
def corrected_resampled_ttest(diffs, n_train, n_test):
    """Nadeau & Bengio corrected t-test cho repeated k-fold CV."""
    diffs = np.asarray(diffs, dtype=float)
    n = len(diffs)
    mean_d = diffs.mean()
    var_d = diffs.var(ddof=1)
    if var_d == 0:
        return (np.inf if mean_d != 0 else 0.0), (0.0 if mean_d != 0 else 1.0)
    corr = (1.0 / n) + (n_test / float(n_train))
    t = mean_d / math.sqrt(var_d * corr)
    p = 2 * sstats.t.sf(abs(t), df=n - 1)
    return float(t), float(p)

def bootstrap_ci(diffs, n_boot=5000, seed=0):
    rng = np.random.RandomState(seed)
    diffs = np.asarray(diffs, dtype=float)
    means = [diffs[rng.randint(0, len(diffs), len(diffs))].mean() for _ in range(n_boot)]
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

def best_cfg(sub, modules):
    m = sub[sub["module"].isin(modules)]
    g = m.groupby(["module", "model"])["f1"].mean().reset_index()
    r = g.sort_values("f1", ascending=False).iloc[0]
    return r["module"], r["model"]

comp_rows = []
n_total = len(ready[DRUGS[0]]["y"])            # xấp xỉ; dùng cho tỉ lệ train/test
n_test_approx = n_total / N_SPLITS
n_train_approx = n_total - n_test_approx

for drug in DRUGS:
    sub = fold_df[fold_df["drug"] == drug]
    b_mod, b_model = best_cfg(sub, ["paper_ready50"])
    a_mod, a_model = best_cfg(sub, ["accessory_chi2", "hybrid_ready_chi2"])

    base = sub[(sub["module"] == b_mod) & (sub["model"] == b_model)].sort_values(["repeat", "fold"])
    adap = sub[(sub["module"] == a_mod) & (sub["model"] == a_model)].sort_values(["repeat", "fold"])
    key = ["repeat", "fold"]
    merged = base.merge(adap, on=key, suffixes=("_base", "_adap"))
    diffs = (merged["f1_adap"] - merged["f1_base"]).values

    t, p = corrected_resampled_ttest(diffs, n_train_approx, n_test_approx)
    try:
        w_stat, w_p = sstats.wilcoxon(merged["f1_adap"], merged["f1_base"])
    except Exception:
        w_stat, w_p = np.nan, np.nan
    lo, hi = bootstrap_ci(diffs, seed=BASE_SEED)

    comp_rows.append({
        "drug": drug,
        "baseline": f"{b_mod}/{b_model}",
        "adaptive_best": f"{a_mod}/{a_model}",
        "f1_base": round(base["f1"].mean(), 4),
        "f1_adaptive": round(adap["f1"].mean(), 4),
        "delta_f1": round(diffs.mean(), 4),
        "ci95_low": round(lo, 4), "ci95_high": round(hi, 4),
        "t_corrected": round(t, 3), "p_corrected": p,
        "wilcoxon_p": w_p,
    })

comp = pd.DataFrame(comp_rows)

# Holm-Bonferroni trên p_corrected
order = comp["p_corrected"].fillna(1.0).values.argsort()
m = len(comp)
holm = np.empty(m); running = 0.0
for rank, idx in enumerate(order):
    adj = min((m - rank) * comp["p_corrected"].fillna(1.0).values[idx], 1.0)
    running = max(running, adj)
    holm[idx] = running
comp["p_holm"] = holm
comp["significant_holm"] = comp["p_holm"] < ALPHA

comp.to_csv(OUTPUT_DIR / "Y_statistical_comparison.csv", index=False)
display(comp)
print("\\nSố thuốc adaptive > baseline có ý nghĩa (Holm < %.2f): %d/%d"
      % (ALPHA, int(comp["significant_holm"].sum()), len(comp)))

## 6. Lineage-aware evaluation (leave-cluster-out)

**Chẩn đoán quan trọng:** accessory genome của bộ này rất đa dạng — Jaccard tối đa giữa hai mẫu chỉ ~0.07. Nếu gộp cụm theo *ngưỡng* Jaccard sẽ ra toàn cụm đơn lẻ, tức leave-cluster-out thoái hóa thành random split (không nói được gì về lineage). Vì vậy ta ép về **K cụm** bằng agglomerative clustering (average linkage) trên khoảng cách Jaccard `1 - sim`, rồi chạy **GroupKFold** (không để cùng cụm nằm cả train lẫn test) cho cấu hình adaptive-best. Mức tụt F1 so với random = dấu hiệu phụ thuộc cấu trúc cụm.

> Đây là proxy dựa trên accessory content, không phải lineage thật (serovar/cgMLST). Nếu có metadata serovar/cgMLST/SNP-distance nên thay `labels` bằng nhãn đó để claim mạnh hơn.


In [ ]:
def jaccard_distance_matrix(Xb):
    Xs = sparse.csr_matrix((Xb > 0).astype(np.uint8))
    inter = (Xs @ Xs.T).toarray().astype(np.float32)
    rs = np.asarray(Xs.sum(axis=1)).ravel().astype(np.float32)
    union = rs[:, None] + rs[None, :] - inter
    sim = np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)
    dist = 1.0 - sim
    np.fill_diagonal(dist, 0.0)
    iu = np.triu_indices(len(dist), 1)
    max_jac = float(sim[iu].max()) if len(iu[0]) else 0.0
    return dist, max_jac

lineage_rows = []
for drug in DRUGS:
    d = ready[drug]
    X_ready, X_acc, y = d["X_ready"], d["X_acc"], np.asarray(d["y"])
    a_mod, a_model = best_cfg(fold_df[fold_df["drug"] == drug],
                              ["accessory_chi2", "hybrid_ready_chi2"])

    dist, max_jac = jaccard_distance_matrix(X_acc.values)
    n_clusters = min(N_LINEAGE_CLUSTERS, len(y) - 1)
    labels = AgglomerativeClustering(
        n_clusters=n_clusters, metric="precomputed", linkage="average").fit_predict(dist)

    # random CV F1 (đã có trong fold_df)
    rnd = fold_df[(fold_df.drug == drug) & (fold_df.module == a_mod) & (fold_df.model == a_model)]["f1"]

    # lineage-aware GroupKFold
    n_splits = min(N_SPLITS, len(np.unique(labels)))
    gkf = GroupKFold(n_splits=n_splits)
    lin_f1 = []
    for tr_idx, te_idx in gkf.split(np.zeros(len(y)), y, groups=labels):
        if len(np.unique(y[te_idx])) < 2 or len(np.unique(y[tr_idx])) < 2:
            continue
        sc = eval_fold(a_mod, a_model, X_ready, X_acc, y, tr_idx, te_idx, seed=BASE_SEED)
        if sc: lin_f1.append(sc["f1"])

    lineage_rows.append({
        "drug": drug, "config": f"{a_mod}/{a_model}",
        "max_pairwise_jaccard": round(max_jac, 4), "n_clusters": int(n_clusters),
        "random_f1_mean": round(float(rnd.mean()), 4),
        "lineage_f1_mean": round(float(np.mean(lin_f1)), 4) if lin_f1 else np.nan,
        "f1_drop": round(float(rnd.mean() - np.mean(lin_f1)), 4) if lin_f1 else np.nan,
    })
    print("Xong lineage cho", drug)

lineage_df = pd.DataFrame(lineage_rows)
lineage_df.to_csv(OUTPUT_DIR / "Y_lineage_aware.csv", index=False)
display(lineage_df)

## 7. Negative control (shuffled labels)

Xáo trộn nhãn rồi chạy lại cấu hình adaptive-best. Kỳ vọng balanced accuracy ≈ 0.5 → pipeline không rò rỉ.


In [ ]:
neg_rows = []
rng = np.random.RandomState(BASE_SEED)
for drug in DRUGS:
    d = ready[drug]
    X_ready, X_acc, y = d["X_ready"], d["X_acc"], np.asarray(d["y"])
    a_mod, a_model = best_cfg(fold_df[fold_df["drug"] == drug],
                              ["accessory_chi2", "hybrid_ready_chi2"])
    y_shuf = y.copy(); rng.shuffle(y_shuf)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=BASE_SEED)
    bas = []
    for tr_idx, te_idx in skf.split(np.zeros(len(y_shuf)), y_shuf):
        sc = eval_fold(a_mod, a_model, X_ready, X_acc, y_shuf, tr_idx, te_idx, seed=BASE_SEED)
        if sc: bas.append(sc["balanced_accuracy"])
    neg_rows.append({"drug": drug, "config": f"{a_mod}/{a_model}",
                     "neg_control_balanced_acc": round(float(np.mean(bas)), 4)})
neg_df = pd.DataFrame(neg_rows)
neg_df.to_csv(OUTPUT_DIR / "Y_negative_control.csv", index=False)
display(neg_df)

## 8. Hình cho báo cáo

In [ ]:
# Fig 1: delta F1 + bootstrap CI
fig, ax = plt.subplots(figsize=(7, 4))
xpos = np.arange(len(comp))
ax.bar(xpos, comp["delta_f1"],
       yerr=[comp["delta_f1"] - comp["ci95_low"], comp["ci95_high"] - comp["delta_f1"]],
       capsize=5, color=["#2a9d8f" if s else "#adb5bd" for s in comp["significant_holm"]])
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(xpos); ax.set_xticklabels(comp["drug"])
ax.set_ylabel("Δ F1 (adaptive − paper_ready50)")
ax.set_title("Adaptive feature fusion vs paper-ready 50 (bootstrap 95% CI)\\nmàu xanh = có ý nghĩa Holm")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "Y_fig1_delta_f1.png", dpi=150); plt.show()

In [ ]:
# Fig 2: random vs lineage-aware F1
fig, ax = plt.subplots(figsize=(7, 4))
w = 0.38; xpos = np.arange(len(lineage_df))
ax.bar(xpos - w/2, lineage_df["random_f1_mean"], w, label="random CV", color="#264653")
ax.bar(xpos + w/2, lineage_df["lineage_f1_mean"], w, label="lineage-aware CV", color="#e76f51")
ax.set_xticks(xpos); ax.set_xticklabels(lineage_df["drug"])
ax.set_ylabel("F1"); ax.set_ylim(0, 1)
ax.set_title("Random vs lineage-aware (leave-cluster-out) evaluation")
ax.legend(); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "Y_fig2_lineage.png", dpi=150); plt.show()

## 9. Gom output + zip

In [ ]:
summary_md = OUTPUT_DIR / "Y_SUMMARY.md"
lines = ["# Direction Y — Robustness & statistical validation\n"]
lines.append("## Statistical comparison (adaptive-best vs paper_ready50)\n")
lines.append(comp.to_markdown(index=False))
lines.append("\n\n## Lineage-aware evaluation\n")
lines.append(lineage_df.to_markdown(index=False))
lines.append("\n\n## Negative control (shuffled labels)\n")
lines.append(neg_df.to_markdown(index=False))
summary_md.write_text("\n".join(lines), encoding="utf-8")

zip_path = str(BASE_DIR / "direction_Y_outputs")
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
print("Đã lưu:", summary_md)
print("Zip:", zip_path + ".zip")
!ls -la "{OUTPUT_DIR}"

# Tải về máy (Colab)
try:
    from google.colab import files
    files.download(zip_path + ".zip")
except Exception as e:
    print("Không ở Colab hoặc bỏ qua download:", e)

## 10. Cách viết vào khóa luận

**RQ mới:** *Lợi thế của accessory-based adaptive feature fusion có bền vững về mặt thống kê và độc lập với cấu trúc dòng không?*

- **Bảng `Y_statistical_comparison.csv`** → báo cáo Δ F1 kèm 95% CI, *t* hiệu chỉnh (Nadeau–Bengio), Wilcoxon, và Holm-adjusted p. Viết: "adaptive fusion tăng F1 có ý nghĩa ở N/5 thuốc sau hiệu chỉnh đa so sánh."
- **Bảng `Y_lineage_aware.csv` + Fig 2** → định lượng mức tụt F1 khi chuyển từ random sang leave-cluster-out. Viết trung thực: điểm tụt bao nhiêu → mức phụ thuộc lineage.
- **Bảng `Y_negative_control.csv`** → balanced accuracy ≈ 0.5 chứng minh không rò rỉ.

Chèn thẳng vào chương *Results / Robustness checks* và mục *Limitations*. Không overclaim: nếu lineage-aware làm tụt mạnh, ghi rõ đó là giới hạn generalization.
